Use SMOTE for address class imbalance

In [1]:
!pip install imbalanced-learn

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv("Customer Churn new.csv")
df.head()

In [4]:
#PReprocessing
# Drop unnecessary columns
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

# Handle missing values
df = df.dropna()

# Encode categorical variables
le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

In [5]:
#Features & Target
X = df.drop('Exited', axis=1)
y = df['Exited']

In [6]:
#Stratified Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.6, stratify=y_temp, random_state=42
)

In [ ]:
#Apply SMOTE (ONLY on training data
smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", pd.Series(y_train_sm).value_counts())

In [8]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train_sm, y_train_sm)

RandomForestClassifier(max_depth=10, n_estimators=200, random_state=42)

In [9]:
y_val_pred = rf_model.predict(X_val)

print("Validation Results:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred))
print("Recall:", recall_score(y_val, y_val_pred))
print("F1 Score:", f1_score(y_val, y_val_pred))

Validation Results:
Accuracy: 0.742
Precision: 0.4075342465753425
Recall: 0.5833333333333334
F1 Score: 0.4798387096774194


In [10]:
y_test_pred = rf_model.predict(X_test)

print("\nTest Results:")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("F1 Score:", f1_score(y_test, y_test_pred))


Test Results:
Accuracy: 0.726
Precision: 0.38009049773755654
Recall: 0.5508196721311476
F1 Score: 0.4497991967871486


In [11]:
y_test_pred = rf_model.predict(X_test)

print("\nTest Results:")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("F1 Score:", f1_score(y_test, y_test_pred))


Test Results:
Accuracy: 0.726
Precision: 0.38009049773755654
Recall: 0.5508196721311476
F1 Score: 0.4497991967871486


In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix:\n", cm)

In [ ]:
print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()